# 02 — Augmentasi Massal + Split Anti-Bocor**Tujuan:** membuat dataset augmentasi skala besar dari wajah ter-align (notebook 01), lalu membagi data ke train/val/test **per grup foto asli** (anti data-leakage).**Prasyarat:** `01_preprocessing_wajah.ipynb` sudah dijalankan → harus ada:- `./data_riset/aug\aligned\<nama_folder_orang>\g<NNNN>.jpg`- `./data_riset/aug\aligned_manifest.csv`**Urutan eksekusi:** SEL 1 → SEL 8, berurutan, dari kernel kosong. Tidak ada variabel dari notebook lain.**Volume target (PRD §10.1):** 680 grup foto asli × (1 asli + 25 augmentasi) = **±17.680 gambar**.**Output (artefak untuk notebook 03–05):**| Artefak | Isi ||---|---|| `Kapita_aug\augmented\<nama_folder_orang>\<grup>_orig.jpg` | wajah ter-align asli (salin, tanpa augmentasi) || `Kapita_aug\augmented\<nama_folder_orang>\<grup>_aug00..24.jpg` | 25 varian augmentasi per grup || `Kapita_aug\augmented_manifest.csv` | person, grup, split, varian, path || `Kapita_aug\split_assignment.csv` | **unit split = grup foto asli** (sumber kebenaran split) |**Aturan yang dijaga (PRD §8.4 no. 1 & §10.3):**1. Split dilakukan **per grup foto asli**, bukan per file — tidak boleh ada grup yang muncul di dua split.2. Augmentasi offline, **deterministik** (seed tetap per grup+varian), pola nama `g0000_orig.jpg`, `g0000_aug00.jpg`.3. Tidak ada transformasi yang memotong wajah keluar frame (batas shift/rotate kecil + `border_mode=BORDER_REFLECT`).4. Albumentations **dipin 1.4.16** — parameter di bawah hanya valid di 1.x.

In [ ]:
# SEL 1 — Impor paket + konstanta
import os
import sys
import time
import zlib
import random
import shutil
from datetime import datetime

# Matikan pengecekan versi daring Albumentations (menghilangkan warning yang tidak berguna)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

import numpy as np
import pandas as pd
import cv2
import torch
import pillow_heif
from PIL import Image
pillow_heif.register_heif_opener()

import albumentations as A
import matplotlib
import matplotlib.pyplot as plt

# --- struktur direktori (PRD §20.B) ---
DATA_ROOT = r"./data_riset/raw"
AUG_ROOT = r"./data_riset/aug"
ALIGN_DIR = os.path.join(AUG_ROOT, "aligned")
ALIGN_MANIFEST = os.path.join(AUG_ROOT, "aligned_manifest.csv")
AUG_DIR = os.path.join(AUG_ROOT, "augmented")
AUG_MANIFEST = os.path.join(AUG_ROOT, "augmented_manifest.csv")
SPLIT_FILE = os.path.join(AUG_ROOT, "split_assignment.csv")

# --- parameter augmentasi (PRD §10) ---
N_AUG = 25                 # varian augmentasi per grup foto asli (rentang izinkan 20-35)
IMG_SIZE = 160
SEED = 42
FRAC_TRAIN, FRAC_VAL = 0.70, 0.15     # sisanya jadi test (PRD §10.3)
FORCE_ALL = False          # True = buat ulang walau file sudah ada

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(AUG_DIR, exist_ok=True)

print("Python            :", sys.version.split()[0])
print("albumentations    :", A.__version__)
print("ALIGN_MANIFEST    :", os.path.isfile(ALIGN_MANIFEST))
print("AUG_DIR           :", AUG_DIR)
print("N_AUG             :", N_AUG, "| total gambar diperkirakan:",
      680 * (1 + N_AUG))

In [ ]:
# SEL 2 — Baca manifest-aligned + validasi prasyarat (harus ada hasil notebook 01)
def imread_unicode(path):
    """Baca gambar (aman path non-ASCII + fallback HEIC)."""
    img = None
    try:
        img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        img = None
    if img is None:
        try:
            with Image.open(path) as pil:
                img = cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)
        except Exception:
            return None
    return img


def imwrite_unicode(path, img, quality=92):
    """Simpan JPG dengan nama file Unicode-safe."""
    ext = os.path.splitext(path)[1]
    params = [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)]
    ok, buf = cv2.imencode(ext, img, params)
    if not ok:
        raise IOError("gagal encode: " + path)
    buf.tofile(path)


assert os.path.isfile(ALIGN_MANIFEST), "jalankan 01_preprocessing_wajah.ipynb lebih dulu"
df_align = pd.read_csv(ALIGN_MANIFEST, encoding="utf-8-sig")
# Buang foto karantina (MTCNN gagal total -> kualitas buruk, PRD §8.4 no. 9)
jumlah_awal = len(df_align)
karantina = df_align[df_align["status"].astype(str).str.startswith("karantina")]
if len(karantina):
    print("foto KARANTINA dibuang:", len(karantina))
    print(karantina[["person", "grup", "nama_file"]].to_string(index=False))
df_align = df_align[df_align["path_aligned"].astype(str).str.len() > 0].copy()
df_align = df_align[~df_align["status"].astype(str).str.startswith("karantina")].reset_index(drop=True)
print("baris setelah buang karantina:", len(df_align), "dari", jumlah_awal)
print()

jumlah_orang = df_align["person"].nunique()
jumlah_grup = df_align.groupby(["person", "grup"]).ngroups
ada_di_disk = sum(1 for p in df_align["path_aligned"] if os.path.isfile(p))

print("baris manifest aligned :", len(df_align))
print("kelas/orang            :", jumlah_orang)
print("grup foto asli         :", jumlah_grup)
print("file aligned di disk   :", ada_di_disk, "/", len(df_align))
print("\nrincian status preprocessing (dari notebook 01):")
print(df_align["status"].value_counts().to_string())

assert jumlah_orang == 69, "harus tetap 69 kelas (dilarang ada yang jatuh <3 foto)"
assert jumlah_grup >= 660, "grup foto asli turun terlalu banyak: {}".format(jumlah_grup)
assert ada_di_disk == len(df_align), "sebagian file aligned hilang -> jalankan ulang notebook 01"
assert not df_align.duplicated(subset=["person", "grup"]).any(), "grup foto duplikat di manifest"
print("\nOK: prasyarat notebook 01 terpenuhi.")

In [ ]:
# SEL 3 — Pipeline augmentasi (PRD §10.2) — WAJIB versi albumentations 1.4.x
transform = A.Compose([
    # --- geometris (batas kecil agar wajah tidak keluar frame) ---
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15,
                       border_mode=cv2.BORDER_REFLECT, p=0.5),
    A.Perspective(scale=(0.02, 0.05), keep_size=True, p=0.3),
    # --- fotometris ---
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.3),
    A.CLAHE(p=0.2),
    A.GaussNoise(p=0.2),
    A.MotionBlur(p=0.2),
    A.ImageCompression(quality_lower=60, quality_upper=100, p=0.3),
    # --- oklusi / kondisi nyata ---
    A.CoarseDropout(max_holes=4, max_height=24, max_width=24, p=0.3),
    A.RandomShadow(p=0.2),
])

N_TRANSFORM = len(transform.transforms)
print("jumlah transform aktif :", N_TRANSFORM)
for t in transform.transforms:
    print("  -", type(t).__name__, "| p =", getattr(t, "p", "?"))
print("\ncatatan: N_AUG =", N_AUG, "varian per grup foto asli (PRD §10.1)")

In [ ]:
# SEL 4 — Split per grup foto asli (anti data-leakage, PRD §10.3)
def buat_split(df, seed=SEED, frac_train=FRAC_TRAIN, frac_val=FRAC_VAL):
    """Tentukan split untuk setiap (person, grup). Stratified per orang agar
    tiap orang punya Train dan (bila memungkinkan) Val & Test sendiri.
    Inti anti-bocor: yang di-split adalah GRUP FOTO ASLI, bukan file hasil augmentasi."""
    rng = np.random.default_rng(seed)
    rows = []
    for person, sub in df.groupby("person"):
        grups = sorted(sub["grup"].unique())
        n = len(grups)
        urut = rng.permutation(n)
        n_train = int(round(frac_train * n))
        n_test = int(round((1.0 - frac_train - frac_val) * n))
        if n >= 4:
            n_train = max(1, min(n_train, n - 2))
            n_test = max(1, min(n_test, n - n_train - 1))
            n_val = max(1, n - n_train - n_test)
        elif n == 3:
            n_train, n_val, n_test = 1, 1, 1
        else:
            n_train, n_val, n_test = n, 0, 0
        label = (["train"] * n_train) + (["val"] * n_val) + (["test"] * n_test)
        for i in range(n):
            rows.append({"person": person, "grup": grups[urut[i]], "split": label[i]})
    out = pd.DataFrame(rows).sort_values(["person", "grup"]).reset_index(drop=True)
    return out


df_split = buat_split(df_align)

# --- verifikasi anti-bocoran ---
# PENTING: kunci kecocokan adalah (person, grup) — nama grup "g0000" dipakai
# oleh SEMUA orang, jadi tidak boleh dikelompokkan per "grup" saja.
gambaran = df_split.groupby(["person", "grup"])["split"].nunique()
assert gambaran.max() == 1, "ada grup foto asli yang masuk >1 split (data leakage!)"
assert set(df_split["split"].unique()) == {"train", "val", "test"}, "split tidak lengkap"
assert len(df_split) == df_align.groupby(["person", "grup"]).ngroups, "jumlah grup tidak cocok"

df_split.to_csv(SPLIT_FILE, index=False, encoding="utf-8-sig")

print("jumlah grup per split :", dict(df_split["split"].value_counts()))
print("rasio                 :", df_split["split"].value_counts(normalize=True).round(3).to_dict())
print("\ncontoh:")
print(df_split.head(10).to_string(index=False))

# berapa file augmentasi yang akan dihasilkan per split
per_split_file = df_split.merge(
    df_align[["person", "grup"]].drop_duplicates(), on=["person", "grup"], how="left")
print("\nperkiraan file per split (1 asli + {} augmentasi per grup):".format(N_AUG))
print({s: int(v * (1 + N_AUG)) for s, v in df_split["split"].value_counts().items()})
print("\nfile split:", SPLIT_FILE)
print("OK: split per grup foto asli, tanpa kebocoran.")

In [ ]:
# SEL 5 — Jalankan augmentasi untuk seluruh grup foto asli
grup_df = df_align.drop_duplicates(subset=["person", "grup"]).reset_index(drop=True)
split_map = dict(zip(zip(df_split["person"], df_split["grup"]), df_split["split"]))

rows = []
t_start = time.time()
total_grup = len(grup_df)

for i, rec in enumerate(grup_df.itertuples(index=False), start=1):
    out_dir = os.path.join(AUG_DIR, rec.person)
    os.makedirs(out_dir, exist_ok=True)
    src = imread_unicode(rec.path_aligned)
    if src is None:
        print("  GAGAL BACA:", rec.path_aligned)
        continue
    if src.shape[0] != IMG_SIZE or src.shape[1] != IMG_SIZE:
        src = cv2.resize(src, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    split = split_map.get((rec.person, rec.grup), "train")

    # 1) salin wajah asli (tanpa augmentasi) sebagai _orig
    p_orig = os.path.join(out_dir, rec.grup + "_orig.jpg")
    if FORCE_ALL or not os.path.isfile(p_orig):
        imwrite_unicode(p_orig, src, quality=95)
    rows.append(dict(person=rec.person, grup=rec.grup, split=split, varian="orig",
                     path_augmented=p_orig, path_aligned=rec.path_aligned,
                     status_preprocessing=rec.status))

    # 2) N_AUG varian augmentasi; seed per (grup, varian) -> deterministik
    for v in range(N_AUG):
        nama_varian = "aug{:02d}".format(v)
        p_aug = os.path.join(out_dir, rec.grup + "_" + nama_varian + ".jpg")
        if FORCE_ALL or not os.path.isfile(p_aug):
            # seed stabil: JANGAN pakai hash() bawaan Python (acak per proses),
            # pakai crc32 yang deterministik lintas sesi.
            k_person = zlib.crc32(rec.person.encode("utf-8"))
            seed = (SEED * 1000003 + (k_person % 1000003) * 101
                    + int(rec.grup[1:]) * 997 + v) % (2 ** 31)
            random.seed(seed)
            np.random.seed(seed % (2 ** 32))
            out = transform(image=src)["image"]
            imwrite_unicode(p_aug, out, quality=92)
        rows.append(dict(person=rec.person, grup=rec.grup, split=split, varian=nama_varian,
                         path_augmented=p_aug, path_aligned=rec.path_aligned,
                         status_preprocessing=rec.status))

    if i % 25 == 0 or i == total_grup:
        elapsed = time.time() - t_start
        eta = elapsed / i * (total_grup - i)
        print("  grup [{}/{}] {:5.1f}%  file dibuat {:6d}  elapsed {:.1f}s  ETA {:.1f}s".format(
            i, total_grup, 100.0 * i / total_grup, len(rows), elapsed, eta))

df_aug = pd.DataFrame(rows)
df_aug.to_csv(AUG_MANIFEST, index=False, encoding="utf-8-sig")

durasi = time.time() - t_start
print("\nselesai dalam {:.1f} detik ({:.1f} ms / file)".format(durasi, 1000.0 * durasi / max(len(rows), 1)))
print("jumlah file tercatat   :", len(df_aug))
print("manifest  :", AUG_MANIFEST)

In [ ]:
# SEL 6 — Verifikasi: jumlah, ukuran, kebocoran split, dan variasi (PRD §19.2 butir 6)
import glob

df = pd.read_csv(AUG_MANIFEST, encoding="utf-8-sig")
JUMLAH_GRUP = int(df.groupby(["person", "grup"]).ngroups)
JUMLAH_KELAS = int(df["person"].nunique())
files_disk = glob.glob(os.path.join(AUG_DIR, "*", "*.jpg"))

print("baris manifest         :", len(df))
print("file .jpg di disk      :", len(files_disk))
print("kelas/orang            :", df["person"].nunique())
print("grup foto asli         :", df.groupby(["person", "grup"]).ngroups)
print("\nfile per split:")
print(df.groupby("split")["path_augmented"].count().to_string())
print("\nvarian per grup (harus seragam):")
print(df.groupby(["person", "grup"]).size().value_counts().to_string())

# 1) jumlah file per grup = 1 + N_AUG
per_grup = df.groupby(["person", "grup"]).size()
assert (per_grup == 1 + N_AUG).all(), "jumlah file per grup tidak = 1 + N_AUG"
assert len(files_disk) == len(df), "jumlah file disk tidak cocok manifest"
assert len(per_grup) == JUMLAH_GRUP, "jumlah grup tidak konsisten: {} vs {}".format(len(per_grup), JUMLAH_GRUP)
assert df["person"].nunique() == JUMLAH_KELAS, "jumlah kelas tidak konsisten"

# 2) anti-bocoran: satu grup hanya boleh punya satu split
split_per_grup = df.groupby(["person", "grup"])["split"].nunique()
assert (split_per_grup == 1).all(), "KEBOCORAN: satu grup punya lebih dari satu split"

# 3) semua kelas muncul di Train (enrollment butuh semua orang)
train_per_orang = df[df["split"] == "train"].groupby("person").size()
print("\nkelas di Train        :", train_per_orang.shape[0], "/", JUMLAH_KELAS)
print("kelas tanpa Train     :", 69 - train_per_orang.shape[0])
assert train_per_orang.shape[0] == JUMLAH_KELAS, "ada kelas yang tidak punya data Train"

# 4) cek ukuran file: sampling acak (numpy RNG supaya reproducible)
rng = np.random.default_rng(SEED)
idx = np.sort(rng.choice(len(df), size=min(40, len(df)), replace=False))
cek = df.iloc[idx]
bad = []
for r in cek.itertuples(index=False):
    im = imread_unicode(r.path_augmented)
    if im is None or im.shape != (IMG_SIZE, IMG_SIZE, 3):
        bad.append((r.path_augmented, None if im is None else im.shape))
print("\ncek acak {} file: rusak = {}".format(len(cek), len(bad)))
for b in bad[:5]:
    print("   -", b)
assert len(bad) == 0, "ada file augmentasi rusak / salah ukuran"

# 5) variasi: file dalam satu grup harus berbeda-beda (bukan salinan)
grup_contoh = cek["grup"].iloc[0] if len(cek) else None
if grup_contoh is not None:
    sub = df[(df["person"] == cek["person"].iloc[0]) & (df["grup"] == grup_contoh)]
    hashes = set()
    for p in sub["path_augmented"]:
        im = imread_unicode(p)
        hashes.add(hash(im.tobytes()))
    print("varian unik pada satu grup (dari {} file): {}".format(len(sub), len(hashes)))
    assert len(hashes) > 1, "augmentasi tidak menghasilkan variasi"

total_mb = sum(os.path.getsize(p) for p in files_disk) / 1024 ** 2
print("\nukuran dataset augmentasi: {:.1f} MB".format(total_mb))
print("OK: semua pemeriksaan SEL 6 lolos.")

In [ ]:
# SEL 7 — Visualisasi: asli vs 8 varian augmentasi
def grid_aug(person, grup=None, n_var=8, cols=9):
    sub = df[(df["person"] == person)]
    if grup is not None:
        sub = sub[sub["grup"] == grup]
    else:
        sub = sub[sub["grup"] == sub["grup"].iloc[0]]
    sub = sub.head(1 + n_var)
    fig, axes = plt.subplots(1, len(sub), figsize=(1.55 * len(sub), 2.0))
    axes = np.atleast_1d(axes)
    for ax, (_, r) in zip(axes, sub.iterrows()):
        im = imread_unicode(r.path_augmented)
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(r.varian, fontsize=7)
        ax.axis("off")
    fig.suptitle("{} - {} (split {})".format(person.split("_")[0], grup if grup else sub["grup"].iloc[0],
                                             sub["split"].iloc[0]), fontsize=9)
    plt.tight_layout()
    plt.show()


for p in sorted(df["person"].unique())[:3]:
    grid_aug(p)

# baris kedua: bandingkan antar kelas pada grup yang sama (mis. g0000)
fig, axes = plt.subplots(1, 6, figsize=(11, 2.1))
for ax, p in zip(axes, sorted(df["person"].unique())[3:9]):
    sub = df[(df["person"] == p) & (df["grup"] == "g0000")]
    im = imread_unicode(sub["path_augmented"].iloc[0])
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(p.split("_")[0][:14], fontsize=7)
    ax.axis("off")
fig.suptitle("Wajah ter-align + 1 varian augmentasi per orang (grup g0000)", fontsize=9)
plt.tight_layout()
plt.show()

print("Total file augmentasi :", len(df))
print("Total kelas/orang     :", df["person"].nunique())
print("Lokasi                :", AUG_DIR)
print("Manifest              :", AUG_MANIFEST)
print("Split                 :", SPLIT_FILE)

In [ ]:
# SEL 8 — Ringkasan
ringkas = {
    "waktu_eksekusi": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "n_aug_per_grup": N_AUG,
    "seed": SEED,
    "frac_train": FRAC_TRAIN,
    "frac_val": FRAC_VAL,
    "jumlah_kelas": int(df["person"].nunique()),
    "jumlah_grup_asli": int(df.groupby(["person", "grup"]).ngroups),
    "jumlah_file": int(len(df)),
    "per_split": {str(k): int(v) for k, v in df["split"].value_counts().items()},
    "ukuran_mb": round(sum(os.path.getsize(p) for p in files_disk) / 1024 ** 2, 1),
}
for k, v in ringkas.items():
    print("{:<20}: {}".format(k, v))

print("\nArtefak untuk notebook 03 (enrollment):")
print("  1. folder augmentasi  :", AUG_DIR)
print("  2. augmented_manifest :", AUG_MANIFEST)
print("  3. split_assignment   :", SPLIT_FILE)
print("\nNotebook 03 memakai HANYA split='train' untuk membuat centroid gallery,")
print("notebook 04 memakai split='val' untuk kalibrasi threshold,")
print("notebook 05 memakai split='test' untuk evaluasi honest (A1).")

# Catatan penting untuk notebook 03–051. **Split dihitung per grup foto asli, bukan per file.** Semua file `g0007_orig.jpg`, `g0007_aug00.jpg`, … `g0007_aug24.jpg` selalu berada di split yang **sama**. Inilah penjaga anti-data-leakage (§8.4 no. 1).2. **Sumber kebenaran split** adalah file `split_assignment.csv` (satu baris per `person` + `grup`). Notebook 03/04/5 harus membacanya, bukan menghitung ulang.3. **Augmentasi tidak melatih model apa pun di Fase 1** (arsitektur = embedding, bukan classifier). Gunanya: (a) memperbanyak variasi embedding saat enrollment sehingga centroid lebih stabil, (b) membuat ambang threshold lebih tahan variasi pencahayaan. Peningkatan accuracy menurun di atas N_AUG ±30 (§10.4).4. **Varian berstatus `orig`** dipakai sebagai pembanding baseline di notebook 05.5. Kalau perlu menambah variasi untuk satu orang tanpa menaikkan N_AUG secara global, cukup tambahkan file `_aug25.jpg` dst. secara manual — split tetap mengikuti grup asalnya.